In [1]:
import hydra
from omegaconf import OmegaConf
print('hydra imported')
import os
import torch
from tqdm.auto import tqdm
from datasets.pfams import PfamDataset
print('dataset class imported')
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer
from transformers import EsmModel

# ignore FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

hydra imported
dataset class imported


In [2]:
# onehot pfam model

output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/'
output_dir += 'outputs/pfam_onehot_1e4_3b474421e59ef501ae348f1f3727057e'

config_path = os.path.join(output_dir, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

# Detect model types
encoder_type, generator_type = ('esm', 'progen2')

best_model_path = os.path.join(output_dir, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not foun   d at {best_model_path}")

encoder = hydra.utils.instantiate(config.encoder)
generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

encoder.load_state_dict(checkpoint['encoder_state_dict'])
generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

encoder.to(device)
generator.to(device)
encoder.eval()
generator.eval();

Some weights of TimeAwareEsmForFlow were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['input_cond_proj.0.bias', 'input_cond_proj.0.weight', 'input_cond_proj.2.bias', 'input_cond_proj.2.weight', 'output_cond_proj.0.bias', 'output_cond_proj.0.weight', 'output_cond_proj.2.bias', 'output_cond_proj.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
pfam_dataset_eval = PfamDataset(data_dir='/orcd/data/omarabu/001/gokul/DistributionEmbeddings/data/pfam', 
                           data_file='eval_pfam_tokenized_data_l64_1e4_small.pt',
                           tokenize=False,
                           set_size=16,
                           max_length=64)

pfam_dataset_train = PfamDataset(data_dir='/orcd/data/omarabu/001/gokul/DistributionEmbeddings/data/pfam', 
                           data_file='pfam_tokenized_data_l64_1e4_small.pt',
                           tokenize=False,
                           set_size=16,
                           max_length=64,
                           base_dir='')

# load pretrained model
esm_model = EsmModel.from_pretrained('facebook/esm2_t6_8M_UR50D', trust_remote_code=True).cuda()

esm_tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D', trust_remote_code=True)
esm_tokenizer.pad_token = '<pad>'
esm_tokenizer.bos_token = '<cls>'
esm_tokenizer.eos_token = '<eos>'

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
train_source_embeddings = {}
p = 0
while len(train_source_embeddings) < 7500:
    batch = pfam_dataset_train[p]
    if batch['source_samples']['idx'] in train_source_embeddings.keys():
        continue
    source_samples = batch['source_samples']
    for key in source_samples.keys():
        if isinstance(source_samples[key], torch.Tensor):
            source_samples[key] = source_samples[key].unsqueeze(0).to(device)
    with torch.no_grad():
        x = esm_model(source_samples['esm_input_ids'].squeeze(0),
                      attention_mask=source_samples['esm_attention_mask'].squeeze(0)).last_hidden_state
        mask = source_samples['esm_attention_mask'].unsqueeze(-1).float()
        source_embedding = (x * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        source_embedding = source_embedding.mean(axis=1)
        train_source_embeddings[batch['source_samples']['idx']] = source_embedding
    p += 1
# train_source_embeddings = torch.cat([train_source_embeddings[k] for k in sorted(train_source_embeddings.keys())], dim=0).cpu().numpy()

In [5]:
train_embs_vec = torch.cat([train_source_embeddings[k] for k in sorted(train_source_embeddings.keys())], dim=0).cpu().numpy()
train_embs_vec.shape

(7500, 320)

In [6]:
from sklearn.neighbors import NearestNeighbors

# fit top 1 nearest neighbor on train embeddings
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(train_embs_vec)

In [7]:
onehot_eval_losses = []

for p in tqdm(range(500)):
    pair = pfam_dataset_eval[p]
    source_samples = pair['source_samples']
    target_samples = pair['target_samples']
    
    # Prepare samples
    for key in source_samples.keys():
        if isinstance(source_samples[key], torch.Tensor):
            source_samples[key] = source_samples[key].unsqueeze(0).to(device)
    for key in target_samples.keys():
        if isinstance(target_samples[key], torch.Tensor):
            target_samples[key] = target_samples[key].unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Compute ESM embeddings
        x_src = esm_model(source_samples['esm_input_ids'].squeeze(0),
                          attention_mask=source_samples['esm_attention_mask'].squeeze(0)).last_hidden_state
        mask_src = source_samples['esm_attention_mask'].unsqueeze(-1).float()
        source_esm_emb = ((x_src * mask_src).sum(1) / mask_src.sum(1).clamp(min=1e-9)).mean(0, keepdim=True)
        
        x_tgt = esm_model(target_samples['esm_input_ids'].squeeze(0),
                          attention_mask=target_samples['esm_attention_mask'].squeeze(0)).last_hidden_state
        mask_tgt = target_samples['esm_attention_mask'].unsqueeze(-1).float()
        target_esm_emb = ((x_tgt * mask_tgt).sum(1) / mask_tgt.sum(1).clamp(min=1e-9)).mean(0, keepdim=True)
        # Find nearest neighbors
        src_nn_idx = nbrs.kneighbors(source_esm_emb.cpu().squeeze(0))[1][0][0]
        tgt_nn_idx = nbrs.kneighbors(target_esm_emb.cpu().squeeze(0))[1][0][0]

        src_nn_idx = sorted(list(train_source_embeddings.keys()))[src_nn_idx]
        tgt_nn_idx = sorted(list(train_source_embeddings.keys()))[tgt_nn_idx]
        
        # Create onehot encodings
        source_onehot = encoder({'idx': torch.tensor([src_nn_idx]).to(device)})
        target_onehot = encoder({'idx': torch.tensor([tgt_nn_idx]).to(device)})
        
        # Compute loss
        output = generator.loss(source_samples, target_samples, source_onehot, target_onehot)
        onehot_eval_losses.append(output.item())

print(f"Onehot model eval loss: {np.mean(onehot_eval_losses):.4f}")
print(f"Onehot model eval loss SEM: {np.std(onehot_eval_losses)/np.sqrt(len(onehot_eval_losses)):.4f}")

  0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [8]:
# any to any pfam model

output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/'
output_dir += 'outputs/pfam_esm_dfm_6c61ec377c95ab55c2b7416301172534'

config_path = os.path.join(output_dir, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

# Detect model types
encoder_type, generator_type = ('esm', 'progen2')

best_model_path = os.path.join(output_dir, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not foun   d at {best_model_path}")

encoder = hydra.utils.instantiate(config.encoder)
generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

encoder.load_state_dict(checkpoint['encoder_state_dict'])
generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

encoder.to(device)
generator.to(device)
encoder.eval()
generator.eval();

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of TimeAwareEsmForFlow were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['input_cond_proj.0.bias', 'input_cond_proj.0.weight', 'input_cond_proj.2.bias', 'input_cond_proj.2.weight', 'output_cond_proj.0.bias', 'output_cond_proj.0.weight', 'output_cond_proj.2.bias', 'output_cond_proj.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
anytoany_eval_losses = []

for p in tqdm(range(500)):
    pair = pfam_dataset_eval[p]
    source_samples = pair['source_samples']
    target_samples = pair['target_samples']
    
    # Prepare samples
    for key in source_samples.keys():
        if isinstance(source_samples[key], torch.Tensor):
            source_samples[key] = source_samples[key].unsqueeze(0).to(device)
    for key in target_samples.keys():
        if isinstance(target_samples[key], torch.Tensor):
            target_samples[key] = target_samples[key].unsqueeze(0).to(device)
    
    with torch.no_grad():
        source_embedding = encoder(source_samples)
        target_embedding = encoder(target_samples)
        output = generator.loss(source_samples, target_samples, 
                               source_embedding, target_embedding)
        anytoany_eval_losses.append(output.item())

print(f"Any-to-any model eval loss: {np.mean(anytoany_eval_losses):.4f}")
print(f"Any-to-any model eval loss SEM: {np.std(anytoany_eval_losses)/np.sqrt(len(anytoany_eval_losses)):.4f}")

  0%|          | 0/500 [00:00<?, ?it/s]

Any-to-any model eval loss: 1.3442
Any-to-any model eval loss SEM: 0.0123


In [14]:
# sample from the model and inspect

for p in range(5):
    pair = pfam_dataset_eval[p]
    source_samples = pair['source_samples']
    target_samples = pair['target_samples']
    
    # Prepare samples
    for key in source_samples.keys():
        if isinstance(source_samples[key], torch.Tensor):
            source_samples[key] = source_samples[key].unsqueeze(0).to(device)
    for key in target_samples.keys():
        if isinstance(target_samples[key], torch.Tensor):
            target_samples[key] = target_samples[key].unsqueeze(0).to(device)
    
    with torch.no_grad():
        source_embedding = encoder(source_samples)
        target_embedding = encoder(target_samples)
        sampled_sequences = generator.sample(source_samples, source_embedding, target_embedding, 
                                            num_samples=1)
        # use the esm tokenizer to decode the sequences
        for i in range(sampled_sequences.size(0)):
            seq_ids = sampled_sequences[i].squeeze(0).cpu().numpy().tolist()
            seq_str = esm_tokenizer.decode(seq_ids, skip_special_tokens=True)
            print(f"Sampled sequence {i+1}: {seq_str}")

        # look at a target sequence
        tgt_ids = target_samples['esm_input_ids'][0][0].cpu().numpy().tolist()
        tgt_str = esm_tokenizer.decode(tgt_ids, skip_special_tokens=True)
        print(f"Target sequence: {tgt_str}")
        print()

Sampled sequence 1: L R G I T S V S N R R E L A F K R Q H V G V V I S R S E M L D D L R L Y R A F G S P Q L N P S Q R S G A G G S R A G I N
Target sequence: C F S M S K K T S D P C P C S L K I C L T K N R N F I A L Q I L S E V P M I E V H G Q Y E E Y L M T V R G E L L D E V D

Sampled sequence 1: G E E D G A N K T N P L L D G A V R L N V V G G P F A A A N V P E G G C S G Y P H F K E Q E
Target sequence: C P Y A G E P N Q W I L I T S Q N D S A Q C V D Y K N L Y S E D P P F G T S N S M K E N K Q H I L C C E K

Sampled sequence 1: N E H E I C D P G K I H I S K D I D K Q R D N G I G W S W
Target sequence: M N L K M C K V C K H N G I V S E E G F D V C I F C D T V F T Q

Sampled sequence 1: D W K A E G I V L I L S L S Y G G L I K E T D R Q T N A G K L A S S V R A F D A S A E K E I T I G F T G P K G S Q A L K
Target sequence: S Q P A P D F M L R D I L E R W A G S D D T L R I V M L A K T E E D K L K Q E V M R L E I R R T E M E M L K E A L R G G

Sampled sequence 1: Q Q F V R L S T V E Q T N P 

In [35]:
from geomloss import SamplesLoss

# Initialize energy distance loss
energy_loss = SamplesLoss("energy", p=2)

energy_distances = []

for p in tqdm(range(100)):  # Evaluate on 100 pairs
    pair = pfam_dataset_eval[p]
    source_samples = pair['source_samples']
    target_samples = pair['target_samples']
    
    # Prepare samples
    for key in source_samples.keys():
        if isinstance(source_samples[key], torch.Tensor):
            source_samples[key] = source_samples[key].unsqueeze(0).to(device)
    for key in target_samples.keys():
        if isinstance(target_samples[key], torch.Tensor):
            target_samples[key] = target_samples[key].unsqueeze(0).to(device)
    
    with torch.no_grad():
        source_embedding = encoder(source_samples)
        target_embedding = encoder(target_samples)
        
        # Sample multiple sequences from the model
        num_samples_to_generate = 16  # Match the set size
        sampled_sequences = generator.sample(source_samples, source_embedding, target_embedding, 
                                            num_samples=num_samples_to_generate).squeeze(0)
        sampled_esm_ids = []
        sampled_attention_masks = []
        for i in range(sampled_sequences.size(0)):
            seq_ids = sampled_sequences[i].squeeze(0).cpu().numpy().tolist()[0]
            seq_str = esm_tokenizer.decode(seq_ids, skip_special_tokens=True)
            # probably this is really silly, but retokenize to get attn mask
            esm_tokens = esm_tokenizer(seq_str, return_tensors='pt', 
                                      padding=True, truncation=True, max_length=64)
            sampled_esm_ids.append(esm_tokens['input_ids'])
            sampled_attention_masks.append(esm_tokens['attention_mask'])
        
        # Stack sampled sequences
        sampled_esm_ids = torch.cat(sampled_esm_ids, dim=0).to(device)
        sampled_attention_mask = torch.cat(sampled_attention_masks, dim=0).to(device)
        
        # Compute ESM embeddings for sampled sequences
        x_sampled = esm_model(sampled_esm_ids, attention_mask=sampled_attention_mask).last_hidden_state
        mask_sampled = sampled_attention_mask.unsqueeze(-1).float()
        sampled_embeddings = (x_sampled * mask_sampled).sum(1) / mask_sampled.sum(1).clamp(min=1e-9)
        
        # Compute ESM embeddings for target sequences
        x_tgt = esm_model(target_samples['esm_input_ids'].squeeze(0),
                         attention_mask=target_samples['esm_attention_mask'].squeeze(0)).last_hidden_state
        mask_tgt = target_samples['esm_attention_mask'].squeeze(0).unsqueeze(-1).float()
        target_embeddings = (x_tgt * mask_tgt).sum(1) / mask_tgt.sum(1).clamp(min=1e-9)
        
        # Compute energy distance between distributions
        energy_dist = energy_loss(sampled_embeddings, target_embeddings)
        energy_distances.append(energy_dist.item())

print(f"Energy distance between generated and target distributions: {np.mean(energy_distances):.4f}")
print(f"Energy distance SEM: {np.std(energy_distances)/np.sqrt(len(energy_distances)):.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Energy distance between generated and target distributions: 3.4899
Energy distance SEM: 0.0425
